# Step 1: 비정형 지식 체계화 및 OKF 에이전트 맛보기
## Google Cloud CE를 위한 온톨로지 스타터팩 핸즈온 시리즈 (Part 1/3)

- **학습 목표**: 비정형 업무 규정(SOP/가이드)에서 100% 환각을 유발하는 단순 벡터 RAG의 한계를 극복하고, Google Cloud **Open Knowledge Format (OKF v0.2)** 규격과 **SHA-256 불변 해시 앵커링**, 그리고 Andrej Karpathy 스타일의 **3-Layer LLM-Wiki**를 구축하여 0% 환각 원천 증거 인출 에이전트를 실습합니다.
- **연계 표준 방법론**: [`Stage 2: 시맨틱 지식 온톨로지 모델링 및 정형화 세부 실행 명세서`](../methodology/02_semantic_modeling_and_okf_ko.md)
- **외부 종속성**: **0개 (파이썬 표준 라이브러리 완결)**

---

### 1. 왜 단순 벡터 RAG는 복합 규정에서 100% 환각을 유발하는가? (Concept)
기업의 비정형 규정 문서는 수많은 예외 조건, 단서 조항(예: "만 65세 이상 시니어 특례 면제"), 복잡한 일할 계산 수식이 얽혀 있습니다:
1. **문맥 단절 (Context Fracture)**: 단서 조항이 본문 청크와 분리되어 벡터 유사도 검색에서 누락됨.
2. **환각 인용 (Hallucinated Citation)**: 그럴듯하지만 원문에 존재하지 않는 페이지 번호나 날조된 요약문을 답변으로 출력함.
3. **감사 불가능 (Unauditable)**: 규정이 변경되었을 때 과거의 어떤 AI 의사결정이 영향을 받았는지 역추적이 불가능함.

**아키텍처 해법: Google OKF v0.2 & 불변 해시 앵커 (`DocChunk`)**:
- 문서를 단순 글자 수로 자르지 않고, 헤더와 표의 시각 구조를 유지하는 **무손실 구조적 청킹**을 수행합니다.
- 각 청크에 **SHA-256 해시 기반 불변 앵커 ID(`anchor_{hash}`)**를 영구 부여합니다.
- 지식을 **3-Layer LLM-Wiki**(`summary.md`, `entities.md`, `concepts.md`, `rules.md`)로 정제하여 LLM이 계층적으로 탐색하도록 격리합니다.


### 2. Step 1 엔드투엔드 처리 플로우 (Flow Architecture)

![Step 1 Architecture Flow](assets/step1_architecture_flow.svg)

<details>
<summary>다이어그램 Mermaid 소스 코드 펼쳐보기 (Click to expand Mermaid source)</summary>

```mermaid
flowchart TD
    subgraph Input ["1. 원천 비정형 가이드"]
        RawDoc["sample_policy_guide.md<br/>(단말 지원금 및 180일 약정 규정)"]
    end

    subgraph Process ["2. 무손실 청킹 & 해시 앵커링"]
        Parser["구조화 파서 (헤더, 표, 단서조항 보존)"]
        Hasher["SHA-256 불변 해시 생성기"]
        Parser --> Hasher
        Hasher --> Chunks["DocChunk (DOC_CHUNK_001 ~ 003)<br/>고유 해시 앵커 부여"]
    end

    subgraph Output ["3. 표준 OKF v0.2 및 3-Layer 위키"]
        OKF["ontology.jsonld (OKF v0.2 번들)"]
        Wiki["Karpathy 3-Layer LLM-Wiki<br/>- summary.md (개요)<br/>- entities.md (엔티티)<br/>- concepts.md (개념)<br/>- rules.md (수학적 룰)"]
        Chunks --> OKF & Wiki
    end

    subgraph Agent ["4. 감사 가능한 해시 그라운딩 에이전트"]
        Query["사용자 질문 ('180일 식스플랜 규정')"]
        Query --> Search["해시 앵커 1:1 증거 검색"]
        Search --> Answer["0% 환각 확정 답변 + SHA-256 원천 증거 인출"]
    end

    Input ==> Process ==> Output ==> Agent
```

</details>


### [실습 1.1] 환경 준비 및 원천 비정형 가이드 로드


In [1]:
import os, sys, json, hashlib

# 스타터팩 루트 디렉토리 설정
starter_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if starter_root not in sys.path:
    sys.path.insert(0, starter_root)

guide_path = os.path.join(starter_root, "data", "input", "01_guides", "sample_policy_guide.md")
with open(guide_path, "r", encoding="utf-8") as f:
    raw_content = f.read()

print(f"[OK] Python Runtime: {sys.version.split()[0]}")
print(f"[OK] Loaded Guide: {guide_path} ({len(raw_content)} chars)")
print("\n--- [원천 비정형 가이드 본문 미리보기] ---")
print(raw_content.strip())


[OK] Python Runtime: 3.13.7
[OK] Loaded Guide: /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/input/01_guides/sample_policy_guide.md (1363 chars)

--- [원천 비정형 가이드 본문 미리보기] ---
# [Enterprise Service Consultation Guide: Terminal & Subscription Policy]
<!-- DOC_CHUNK_001: Overview & Scope -->
## 1. General Principles
This document defines standard commercial terms, device subsidy allocations, 180-day baseline plan commitment rules, and statutory difference penalty settlement policies for consumer subscriptions.

<!-- DOC_CHUNK_002: Baseline Plan Commitment (Six-Plan Rule) -->
## 2. 180-Day Baseline Plan Commitment Rule
1. Subscribers receiving public device subsidies must maintain their initial subscription tier at or above the designated Baseline Plan Threshold (e.g., 5G 45,000 KRW/month or LTE 26,400 KRW/month) for a mandatory period of **180 days** from the activation date.
2. If a subscriber downgrades below the baseline threshold within 180 days, a subsidy dif

### [실습 1.2] 결정론적 SHA-256 해시 앵커링 및 OKF v0.2 컴파일 (`pipeline.ingest`)


In [2]:
from pipeline.ingest import run_ingest

# Ingest 파이프라인 실행
run_ingest(guide_path=guide_path, output_dir=os.path.join(starter_root, "data", "output"), wiki_dir=os.path.join(starter_root, "data", "llm_wiki"))

# 생성된 OKF v0.2 번들 로드
onto_path = os.path.join(starter_root, "data", "output", "ontology.jsonld")
with open(onto_path, "r", encoding="utf-8") as f:
    ontology = json.load(f)

print("=== [Google Cloud OKF v0.2 번들 메타데이터] ===")
print(f"@context: {ontology['@context']}")
print(f"Domain ID: {ontology['@graph'][0]['@id']}")
print(f"총 청크 수: {ontology['@graph'][0]['totalChunks']}")
print("\n=== [생성된 불변 해시 앵커 노드 (DocChunk)] ===")
for node in ontology['@graph'][1:]:
    print(f" - ID: {node['@id']:<20} | Title: {node['title']:<35} | Hash Anchor: {node['hashAnchor']}")


[INGEST SUCCESS] Processed 3 chunks -> /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/output/doc_chunks.jsonl, /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/output/ontology.jsonld
[LLM-WIKI SUCCESS] Generated Karpathy 3-Layer Wiki in /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/llm_wiki/
=== [Google Cloud OKF v0.2 번들 메타데이터] ===
@context: https://cloud.google.com/ontology/v0.2/context.jsonld
Domain ID: domain:TelecomSubsidyManagement
총 청크 수: 3

=== [생성된 불변 해시 앵커 노드 (DocChunk)] ===
 - ID: chunk:DOC_CHUNK_001  | Title: Overview & Scope                    | Hash Anchor: anchor_a52e1a72869f
 - ID: chunk:DOC_CHUNK_002  | Title: Baseline Plan Commitment (Six-Plan Rule) | Hash Anchor: anchor_3003b683cea0
 - ID: chunk:DOC_CHUNK_003  | Title: Penalty Math Formula                | Hash Anchor: anchor_47971ec8e4d8


### [실습 1.3] Karpathy 3-Layer LLM-Wiki 구조 검증


In [3]:
wiki_dir = os.path.join(starter_root, "data", "llm_wiki")
wiki_files = ["summary.md", "entities.md", "concepts.md", "rules.md"]

print("=== [Karpathy 3-Layer LLM-Wiki 생성 확인] ===")
for wf in wiki_files:
    p = os.path.join(wiki_dir, wf)
    sz = os.path.getsize(p)
    with open(p, "r", encoding="utf-8") as f:
        first_line = f.readline().strip()
    print(f" - Layer File: {wf:<15} ({sz:>5} bytes) -> {first_line}")

print("\n=== [Layer 3: rules.md 내 비즈니스 수식 발췌] ===")
with open(os.path.join(wiki_dir, "rules.md"), "r", encoding="utf-8") as f:
    r_content = f.read()
for line in r_content.splitlines():
    if "Penalty" in line or "180" in line or "Exemption" in line or "면제" in line:
        print(f"  > {line}")


=== [Karpathy 3-Layer LLM-Wiki 생성 확인] ===
 - Layer File: summary.md      (  561 bytes) -> # Enterprise Knowledge Wiki: Executive Summary (Layer 1)
 - Layer File: entities.md     (  831 bytes) -> # Enterprise Knowledge Wiki: Candidate Entities Directory (Layer 2)
 - Layer File: concepts.md     (  804 bytes) -> # Enterprise Knowledge Wiki: 78 Semantic Concepts & Taxonomies (Layer 3)
 - Layer File: rules.md        (  956 bytes) -> # Enterprise Knowledge Wiki: Codified Business Rules (Layer 3)

=== [Layer 3: rules.md 내 비즈니스 수식 발췌] ===
  > ### Rule 1: 180-Day Six-Plan Commitment Obligation (`RULE_SIX_PLAN_180D`)
  > - **Constraint**: Must maintain baseline tier or higher for at least 180 days from activation date.
  > - **Penalty on Breach (< 180 Days)**:
  >   `Penalty = (Initial Subsidy - Downgraded Plan Subsidy) * (180 - Maintained Days) / 180`
  > - **Post-180 Days**: Differential clawback is completely waived (0 KRW penalty).


### [실습 1.4] 불변 해시 앵커 기반 원천 증거 인출 에이전트 구동


In [4]:
chunks_path = os.path.join(starter_root, "data", "output", "doc_chunks.jsonl")
chunks = []
with open(chunks_path, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

def retrieve_policy_with_anchor(keyword: str):
    matched = [c for c in chunks if keyword in c["content"] or keyword in c["title"]]
    return [{
        "chunk_id": c["chunk_id"],
        "title": c["title"],
        "hash_anchor": c["hash_anchor"],
        "content_snippet": c["content"].replace("\n", " ")[:150] + "..."
    } for c in matched]

print("=== [에이전트 인출 테스트: 질의어 '180일 식스플랜 / 위약금'] ===")
results = retrieve_policy_with_anchor("180")
for item in results:
    print(f"✔ [검증된 출처] 청크: {item['chunk_id']} | 불변 해시: {item['hash_anchor']}")
    print(f"  제목: {item['title']}")
    print(f"  발췌 원문: {item['content_snippet']}\n")


=== [에이전트 인출 테스트: 질의어 '180일 식스플랜 / 위약금'] ===
✔ [검증된 출처] 청크: DOC_CHUNK_001 | 불변 해시: anchor_a52e1a72869f
  제목: Overview & Scope
  발췌 원문: ## 1. General Principles This document defines standard commercial terms, device subsidy allocations, 180-day baseline plan commitment rules, and stat...

✔ [검증된 출처] 청크: DOC_CHUNK_002 | 불변 해시: anchor_3003b683cea0
  제목: Baseline Plan Commitment (Six-Plan Rule)
  발췌 원문: ## 2. 180-Day Baseline Plan Commitment Rule 1. Subscribers receiving public device subsidies must maintain their initial subscription tier at or above...

✔ [검증된 출처] 청크: DOC_CHUNK_003 | 불변 해시: anchor_47971ec8e4d8
  제목: Penalty Math Formula
  발췌 원문: ## 3. Penalty Settlement Computation Formula The subsidy difference penalty is calculated according to the following deterministic formula: `Penalty =...



### 3. 결과 해석 및 실무적 의미 (Interpretation)
- **불변 해시의 신뢰성**: 각 청크의 본문이 1글자라도 변경되면 `hash_anchor`가 즉시 변경되므로, 규정 위변조를 즉각 감지할 수 있습니다.
- **LLM-Wiki의 인지 부하 감소**: LLM이 전체 가이드 문서를 통째로 읽는 대신 `summary.md` -> `rules.md` 순서로 필요한 레이어만 선택적으로 인출하여 토큰을 절약하고 환각을 원천 차단합니다.
- **Step 2 연계**: 여기서 정제된 `DOC_CHUNK_001~003`과 `DomainRule`은 Step 2에서 BigQuery Property Graph의 물리 노드로 직접 바인딩됩니다.
